In [ ]:
import ollama

In [ ]:
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import TextLoader

In [ ]:
from rich.console import Console
from rich.panel import Panel
from rich.pretty import Pretty
from rich.live import Live
console = Console()

In [ ]:
model = "qwen3:4b"
temprature = 0.0

In [ ]:
doc = Document(
    page_content="This is the main text content I am using to create a RAG1",
    metadata = {
        "author" : "Advait Kale"
    }
)
doc

In [ ]:
import os
os.makedirs("data/text_files", exist_ok=True)

In [ ]:
loader = TextLoader(
    file_path="data/text_files/python_intro.txt",
    encoding="utf-8"
)
loader

In [ ]:
loader.load()


## RAG Pipelines

In [ ]:
from pathlib import Path
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import TextLoader, PyPDFLoader, PyMuPDFLoader


def process_all_pdfs(path: str):

    all_docs = []
    pdf_path = Path(path)
    pdf_files = list(pdf_path.glob("*.pdf"))

    # just checking the glob
    console.print(Panel(Pretty(pdf_files)))
    print(type(pdf_files))
    #getting the contents

    for pdf_i in pdf_files:
        print(pdf_i.name)
        try:
            loader = PyPDFLoader(file_path=str(pdf_i))
            docs = loader.load()
            for j in docs:
                j.metadata["source_file"] = pdf_i.name
                j.metadata["file_type"] = "pdf"
            all_docs.extend(docs)
            # console.print(Panel(Pretty(all_docs)))
        except Exception as e:
            print(f"Error {e}")
    return all_docs



process_all_pdfs("C:\Advait\VS_Code\VS code 2.0\RAG_YT\RAG_from_Scratch\data\pdfs")

In [ ]:
all_pdf_docs = process_all_pdfs(r"C:\Advait\VS_Code\VS code 2.0\RAG_YT\RAG_from_Scratch\data\pdfs")

In [ ]:
all_pdf_docs

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_into_chunks(documents, chunk_size : int = 1000, chunk_overlap: int  = 150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        separators=["\n\n", "\n", " ", ".", ""]
    )
    split_docs = splitter.split_documents(documents = documents)
    return split_docs

chunks = split_into_chunks(all_pdf_docs)
chunks

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from typing import List, Tuple, Any, Dict
from sklearn.metrics.pairwise import cosine_similarity

embedding_model = "all-MiniLM-L6-v2"

class EmbeddingManager:
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Embedding Dimention {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"error {e}")

    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        
        if not self.model:
            raise ValueError("Model Not Loaded")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings


In [ ]:
embedding_manager = EmbeddingManager(embedding_model)
embedding_manager

## vector store

In [ ]:
import uuid

class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", dir: str = "data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = dir
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):

        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # FIX: delete any existing collection first, then recreate as COSINE.
            # A collection's distance metric is locked at creation, so an old
            # default-L2 one must be dropped before cosine can take. Also clears
            # the duplicate rows piled up from earlier re-runs.
            try:
                self.client.delete_collection(self.collection_name)
            except Exception:
                pass

            self.collection = self.client.create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine",   # cosine distance, not default L2
                }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"space: {self.collection.metadata.get('hnsw:space')}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

In [ ]:
texts = [doc.page_content for doc in chunks ]
embedding = embedding_manager.generate_embedding(texts)

vectorstore.add_documents(chunks, embeddings=embedding)

In [ ]:
class RAGRetrieve:
    def __init__(self, vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_thres: float = None):

        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        try:
            results = self.vectorstore.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results=top_k
            )
            retrievd_doc = []
            if results['documents'] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, dist) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - dist

                    # FIX: by default keep all nearest docs (query already returns
                    # the closest top_k). Only drop if a threshold is explicitly set.
                    if score_thres is None or similarity_score >= score_thres:
                        retrievd_doc.append({
                            "id" : doc_id,
                            "content" : document,
                            "metadata" : metadata,
                            "similarity" : similarity_score,
                            "distance" : dist,
                            "rank" : i + 1
                        })
            return retrievd_doc
        except Exception as e:
            print(f"Error {e}")
            return []


Rag_retriever = RAGRetrieve(vectorstore=vectorstore, embedding_manager=embedding_manager)
Rag_retriever 

In [ ]:
Rag_retriever.retrieve(query="Gita")

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

model = OllamaLLM(model="llama3.1")

chain = prompt | model

chain.invoke({"question": "What is LangChain?"})

In [ ]:
model

In [ ]:
from langchain_ollama import OllamaLLM
llm_model = OllamaLLM(model=model)

In [ ]:
llm = init_chat_model(
    model=model,
    model_provider="ollama",
    temprature = temprature
)

In [ ]:
llm

In [ ]:
def answer(query: str, k: int = 2):
    context_chunks = retrieve(query, 2)
    context_string = [f"<context> {c} >/context>" for c in context_chunks]
    for chunk in llm.stream(RAG_PROMPT.format(context = context_string, query = query)):
        yield chunk.content

In [ ]:
def rag_simple(query, RAGRetrieve, llm, top_k: int= 3):
    results = RAGRetrieve.retrieve(query, top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        return "No relevent context found"
    
    prompt = """ Use the following context to answer the query
    <context>
    {context}
    </context>

    <query>
    {query}
    </query>

    Answer: 
    """

    ans = llm.invoke(prompt.format(context = context, query = query))
    return ans.content
    



In [ ]:
query = "how many chapters are there in the index"

In [ ]:
answer = rag_simple(query=query, RAGRetrieve=Rag_retriever,llm=llm)
answer

In [ ]:
console.print(Panel(Pretty(answer)), width=70)